# StreamView Analytics · Análisis exploratorio (EA3)
Sigue la estrategia analítica de la guía 3.1.1: problema → datos → calidad → variables → relaciones → patrones → hipótesis → decisión.

Ejecutar desde la raíz del proyecto (`jupyter lab` y abrir este archivo).

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append('.')
import pandas as pd, numpy as np
import plotly.express as px
from src.common import separar, explotar
from src.data.clean_data import main as limpiar
if not os.path.exists('data/processed/catalogo_limpio.csv'):
    limpiar()
cat = pd.read_csv('data/processed/catalogo_limpio.csv')
fin = pd.read_csv('data/processed/catalogo_financiero.csv')
print(len(cat), 'títulos |', len(fin), 'películas con datos financieros')

31991 títulos | 3540 películas con datos financieros


## 1-2. Problema y datos disponibles
**Decisión a apoyar:** qué contenido priorizar en adquisición y producción. **Usuarios:** Directorio (ejecutivo) y Gerencia de Contenidos (analítico). Clave única: `uid` (`show_id` se repite entre películas y series).

In [2]:
cat.dtypes.to_frame('tipo').join(cat.isna().sum().rename('nulos'))

,tipo,nulos
show_id,int64,0
type,str,0
title,str,0
director,str,0
cast,str,0
country,str,0
date_added,str,0
release_year,int64,0
genres,str,0
language,str,0


## 4. Calidad de los datos
Un 0 en votos, presupuesto o ingresos significa *dato no disponible*. La calificación solo se considera con ≥ 10 votos.

In [3]:
calidad = pd.DataFrame({'% faltante': [cat['calificacion'].isna().mean()*100, (cat['country']=='Desconocido').mean()*100, (cat['director']=='Desconocido').mean()*100]}, index=['calificacion (<10 votos)','country','director']).round(1)
calidad

,% faltante
calificacion (<10 votos),36.4
country,7.1
director,34.7


In [4]:
pd.read_csv('reports/data_quality/cleaning_summary.csv').head(20)

,campo,accion,justificacion
0,rating (TV),eliminada,idéntica a vote_average en 100% de los casos
1,rating (Movies),eliminada,idéntica a vote_average en 100% de los casos
2,duration (Movies),eliminada,100% de valores nulos
3,date_added,convertida,texto -> datetime
4,show_id duplicados (TV),9 eliminados,keep='first'
5,show_id duplicados (Movies),0 eliminados,keep='first'
6,director (nulos),TV: 10960 / Movies: 132,rellenados con 'Desconocido'
7,cast (nulos),TV: 1156 / Movies: 204,rellenados con 'Desconocido'
8,country (nulos),TV: 1795 / Movies: 466,rellenados con 'Desconocido'
9,genres (nulos),TV: 972 / Movies: 107,rellenados con 'Desconocido'


## 5. Variables individuales

In [5]:
px.histogram(cat.dropna(subset=['calificacion']), x='calificacion', color='type', barmode='overlay', nbins=40, opacity=.65, title='Distribución de calificación')

In [6]:
px.box(cat[cat.popularity>0], x='type', y='popularity', log_y=True, color='type', title='Popularidad por tipo (escala log)')

## 6. Relaciones entre variables
Spearman por tipo: robusta a valores extremos y a escalas distintas.

In [7]:
for t,g in cat.dropna(subset=['calificacion']).groupby('type'):
    print(t, round(g.popularity.corr(g.calificacion, method='spearman'),2), f'(n={len(g):,})')

Movie 0.23 (n=14,183)
TV Show 0.17 (n=6,148)


## 7. Patrones: géneros y países

In [8]:
cat['lg']=cat.genres.map(separar); g=explotar(cat,'lg','genero')
tg=g.groupby('genero').agg(titulos=('uid','size'),calificacion=('calificacion','mean'),popularidad_pct=('popularity_pct','mean')).query('titulos>=300').round(2)
tg.sort_values('popularidad_pct',ascending=False).head(8)

,titulos,calificacion,popularidad_pct
genero,,,
Soap,631,7.14,0.73
Adventure,1768,6.47,0.64
Talk,654,6.55,0.62
Action,3239,6.25,0.61
Fantasy,1450,6.51,0.61
Science Fiction,1454,6.13,0.59
Family,3014,6.85,0.58
Kids,835,7.28,0.55


In [9]:
cat['lp']=cat.country.map(separar); p=explotar(cat,'lp','pais'); p=p[p.pais!='Desconocido']
tp=p.groupby('pais').agg(titulos=('uid','size'),calificacion=('calificacion','mean'),popularidad_pct=('popularity_pct','mean')).query('titulos>=100').round(2)
tp.sort_values('calificacion',ascending=False).head(8)

,titulos,calificacion,popularidad_pct
pais,,,
Japan,2869,7.22,0.47
Colombia,132,7.21,0.58
Egypt,214,7.20,0.41
Mexico,568,7.10,0.60
South Korea,2174,7.10,0.52
Turkey,407,7.08,0.52
China,2399,6.92,0.47
Thailand,304,6.91,0.38


## 7b. Anomalías y análisis financiero
Tres definiciones de ROI (no mezclar).

In [10]:
real=fin[fin.presupuesto_realista]
pd.Series({'ROI agregado (ingresos/presupuesto totales)':fin.revenue.sum()/fin.budget.sum(),'ROI mediano':fin.roi.median(),'ROI promedio (presupuesto >= 10.000)':real.roi.mean(),'ROI promedio sin filtrar (distorsionado)':fin.roi.mean(),'% películas que no recuperan inversión':(fin.roi<1).mean()*100}).round(2)

ROI agregado (ingresos/presupuesto totales)      2.92
ROI mediano                                      1.70
ROI promedio (presupuesto >= 10.000)             4.19
ROI promedio sin filtrar (distorsionado)       781.55
% películas que no recuperan inversión          36.67
dtype: float64

In [11]:
px.scatter(fin, x='budget', y='revenue', log_x=True, log_y=True, color='calificacion', hover_name='title', title='Presupuesto vs. ingresos')

## 8-9. Hipótesis y decisión
- **H1** La popularidad no es sustituto de la calidad (Spearman 0,17-0,23).
- **H2** Japón y Corea del Sur combinan volumen y buena nota; Colombia y México rinden en popularidad con poco volumen.
- **H3** Las películas de bajo presupuesto con ROI atípico son oportunidades **si** el dato se valida.
- **Decisión:** ver `docs/conclusiones_recomendaciones.md`.

*Nota:* la serie temporal es plana (≈2.000 títulos/año): no hay tendencia que interpretar.